### Softmax Regression on MNIST

#### Implemention of softmax regression (multinomial logistic regression)

In [3]:
import time
import torch
import torch.nn.functional as F
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision import transforms

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [6]:
train_dataset = datasets.MNIST(root='data',
               train=True,
               transform=transforms.ToTensor(),
               download=True)

test_dataset = datasets.MNIST(root='data',
               train=False,
               transform=transforms.ToTensor()
               )

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=256,
                          shuffle=True)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=256,
                         shuffle=False)

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 9912422/9912422 [00:23<00:00, 427875.14it/s] 


Extracting data\MNIST\raw\train-images-idx3-ubyte.gz to data\MNIST\raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 28881/28881 [00:00<00:00, 43928.40it/s]


Extracting data\MNIST\raw\train-labels-idx1-ubyte.gz to data\MNIST\raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 1648877/1648877 [00:05<00:00, 278457.99it/s]


Extracting data\MNIST\raw\t10k-images-idx3-ubyte.gz to data\MNIST\raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 4542/4542 [00:00<00:00, 369360.93it/s]

Extracting data\MNIST\raw\t10k-labels-idx1-ubyte.gz to data\MNIST\raw



In [21]:
for images, labels in train_loader:
    print(f"Image batch dimensions: {images.shape} #NCHW")
    print(f"(N)Number of training examples(aka; batch size): {images.size(0)}")
    print(f"(C)Number of channels: {images.size(1)}")
    print(f"(H)Height X (W)Width of an Image: {images.size(2)} X {images.size(3)}")
    print(f"Image label dimensions: {labels.shape}")
    break

Image batch dimensions: torch.Size([256, 1, 28, 28]) #NCHW
(N)Number of training examples(aka; batch size): 256
(C)Number of channels: 1
(H)Height X (W)Width of an Image: 28 X 28
Image label dimensions: torch.Size([256])


In [22]:
# There are images from 1 - 9 letters written.
torch.unique(labels)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [23]:
class SoftmaxRegression(torch.nn.Module):

    def __init__(self, num_features, num_classes):
        super(SoftmaxRegression, self).__init__()
        self.linear = torch.nn.Linear(num_features, num_classes)

        self.linear.weight.detach().zero_()
        self.linear.bias.detach().zero_()

    def forward(self, x):
        logits = self.linear(x)
        probs = F.softmax(logits, dim=1)
        return logits, probs
    

model = SoftmaxRegression(num_features=28*28,
                          num_classes=10)

model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [27]:
torch.manual_seed(123)

def computer_accuarcy(model, data_loader):
    correct_pred, num_examples = 0, 0

    for features, labels in data_loader:
        features = features.view(-1, 28*28).to(device)
        labels = labels.to(device)
        logits, probs = model(features)
        _, predicted_labels = torch.max(probs, 1)
        num_examples += labels.size(0)
        correct_pred += (predicted_labels == labels).sum()
    return correct_pred.float() / num_examples * 100

In [29]:
start_time = time.time()
epoch_cost = []
for epoch in range(25):
    avg_cost = 0.0
    for batch_idx, (features, labels) in enumerate(train_loader):

        features = features.view(-1, 28*28).to(device)
        labels = labels.to(device)

        logits, probs = model(features)
        cost = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        cost.backward()
        avg_cost += cost

        optimizer.step()

        if not batch_idx % 50:
            print(f"Epoch: {epoch+1:03d}/{25:03d} | Batch {batch_idx:03d}/{len(train_loader)} | Cost: {cost:.4f}")

    with torch.set_grad_enabled(False):
        avg_cost = avg_cost / len(train_loader)
        epoch_cost.append(avg_cost)
        print(f"Epoch: {epoch+1:03d}/{25:03d} Train Cost: {avg_cost:.4f}")
        print(f"Time elapsed: {(time.time() - start_time)/60:.2f} min")


Epoch: 001/025 | Batch 000/235 | Cost: 0.4294
Epoch: 001/025 | Batch 050/235 | Cost: 0.3388
Epoch: 001/025 | Batch 100/235 | Cost: 0.3991
Epoch: 001/025 | Batch 150/235 | Cost: 0.4408
Epoch: 001/025 | Batch 200/235 | Cost: 0.4155
Epoch: 001/025 Train Cost: 0.3814
Time elapsed: 0.09 min
Epoch: 002/025 | Batch 000/235 | Cost: 0.3713
Epoch: 002/025 | Batch 050/235 | Cost: 0.4125
Epoch: 002/025 | Batch 100/235 | Cost: 0.3605
Epoch: 002/025 | Batch 150/235 | Cost: 0.3670
Epoch: 002/025 | Batch 200/235 | Cost: 0.2756
Epoch: 002/025 Train Cost: 0.3594
Time elapsed: 0.18 min
Epoch: 003/025 | Batch 000/235 | Cost: 0.3493
Epoch: 003/025 | Batch 050/235 | Cost: 0.3662
Epoch: 003/025 | Batch 100/235 | Cost: 0.2102
Epoch: 003/025 | Batch 150/235 | Cost: 0.3594
Epoch: 003/025 | Batch 200/235 | Cost: 0.3490
Epoch: 003/025 Train Cost: 0.3449
Time elapsed: 0.27 min
Epoch: 004/025 | Batch 000/235 | Cost: 0.3965
Epoch: 004/025 | Batch 050/235 | Cost: 0.3580
Epoch: 004/025 | Batch 100/235 | Cost: 0.3483
E

In [31]:
import plotly.express as px

fig = px.line(x=range(1, len(epoch_cost)+1), y=epoch_cost, title='Training Loss')
fig.update_xaxes(title_text='Epochs')
fig.update_yaxes(title_text='Cross Entropy')
fig.show()

In [32]:
print(f"Test accuracy: {computer_accuarcy(model, test_loader):.2f}%")

Test accuracy: 92.29%


In [34]:
for features, labels in test_loader:
    break

fig = px.imshow(features[0].view(28, 28).cpu().numpy(), color_continuous_scale='gray')
fig.show()

In [35]:
_, predictions = model.forward(features[0].view(-1, 28*28).to(device))
predictions = torch.argmax(predictions, dim=1)
print(f"Model prediction: {predictions.item()}")

Model prediction: 7
